In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

stopwords = {
    "a", "an", "the", "and", "or", "but", "if", "while", "with", "without", "of", "at", "by", "for", "to", "from",
    "in", "on", "into", "onto", "up", "down", "out", "over", "under", "again", "further", "then", "once", "is", "are",
    "was", "were", "be", "been", "being", "am", "do", "does", "did", "doing", "have", "has", "had", "having", "this",
    "that", "these", "those", "it", "its", "as", "not", "no", "nor", "so", "than", "too", "very", "can", "will", "just"
}

def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

def content_tokens(text):
    toks = tokenize(text)
    return [t for t in toks if t not in stopwords and len(t) > 1]

def jaccard_overlap(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def overlap_count(a, b):
    return len(set(a) & set(b))

df["tokens1"] = df["sentence1"].map(content_tokens)
df["tokens2"] = df["sentence2"].map(content_tokens)
df["len1_words"] = df["tokens1"].map(len)
df["len2_words"] = df["tokens2"].map(len)
df["avg_words"] = (df["len1_words"] + df["len2_words"]) / 2.0
df["len_gap_words"] = (df["len1_words"] - df["len2_words"]).abs()
df["lexical_jaccard"] = [jaccard_overlap(a, b) for a, b in zip(df["tokens1"], df["tokens2"])]
df["shared_content_words"] = [overlap_count(a, b) for a, b in zip(df["tokens1"], df["tokens2"])]

base_df = df[(df["len1_words"] >= 4) & (df["len2_words"] >= 4)].copy()

hard_negative_df = base_df[
    (base_df["label"] <= 2.0)
    & (base_df["lexical_jaccard"] >= 0.35)
    & (base_df["shared_content_words"] >= 2)
].copy()

hard_positive_df = base_df[
    (base_df["label"] >= 4.0)
    & (base_df["lexical_jaccard"] <= 0.20)
].copy()

hard_negative_df = hard_negative_df.sort_values(
    by=["lexical_jaccard", "shared_content_words", "label", "avg_words", "sentence1", "sentence2"],
    ascending=[False, False, True, True, True, True],
).head(150)

hard_positive_df = hard_positive_df.sort_values(
    by=["lexical_jaccard", "shared_content_words", "label", "avg_words", "sentence1", "sentence2"],
    ascending=[True, True, False, True, True, True],
).head(150)

subset_df = pd.concat([hard_negative_df, hard_positive_df], axis=0)
subset_df = subset_df.drop_duplicates(subset=["sentence1", "sentence2", "label"])
subset_df = subset_df.sort_values(
    by=["label", "lexical_jaccard", "shared_content_words", "sentence1", "sentence2"],
    ascending=[True, False, False, True, True],
).reset_index(drop=True)

print({
    "original_num_examples": len(df),
    "base_num_examples": len(base_df),
    "hard_negative_candidates": len(hard_negative_df),
    "hard_positive_candidates": len(hard_positive_df),
    "subset_num_examples": len(subset_df),
    "columns": [c for c in subset_df.columns if c not in ["tokens1", "tokens2"]],
})
print(subset_df[["sentence1", "sentence2", "label", "lexical_jaccard", "shared_content_words", "len1_words", "len2_words"]].head(12))


In [ ]:
def overlap_bucket(x):
    if x < 0.10:
        return "low_overlap"
    if x < 0.30:
        return "medium_overlap"
    return "high_overlap"

subset_df["overlap_bucket"] = subset_df["lexical_jaccard"].map(overlap_bucket)

summary_stats = {
    "label_mean": float(subset_df["label"].mean()),
    "label_median": float(subset_df["label"].median()),
    "lexical_jaccard_mean": float(subset_df["lexical_jaccard"].mean()),
    "lexical_jaccard_median": float(subset_df["lexical_jaccard"].median()),
    "shared_content_words_mean": float(subset_df["shared_content_words"].mean()),
    "len1_words_mean": float(subset_df["len1_words"].mean()),
    "len2_words_mean": float(subset_df["len2_words"].mean()),
    "overlap_bucket_counts": subset_df["overlap_bucket"].value_counts().to_dict(),
    "label_band_counts": {
        "low_label_le_2": int((subset_df["label"] <= 2.0).sum()),
        "mid_label_2_to_4": int(((subset_df["label"] > 2.0) & (subset_df["label"] < 4.0)).sum()),
        "high_label_ge_4": int((subset_df["label"] >= 4.0).sum()),
    },
}
print(summary_stats)


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)


In [ ]:
def score_band(x):
    if x < 2.0:
        return "low"
    if x < 4.0:
        return "medium"
    return "high"

pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = subset_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["label_band"] = results_df["label"].map(score_band)
results_df["pred_band"] = results_df["predicted_score_0_5"].map(score_band)
results_df["band_match"] = results_df["label_band"] == results_df["pred_band"]

largest_errors_df = results_df.sort_values("absolute_error", ascending=False).reset_index(drop=True)
high_overlap_errors_df = results_df[results_df["overlap_bucket"] == "high_overlap"].sort_values(
    ["absolute_error", "lexical_jaccard", "shared_content_words"],
    ascending=[False, False, False],
).reset_index(drop=True)

confusion_counts = pd.crosstab(results_df["label_band"], results_df["pred_band"])
confusion_counts = confusion_counts.reindex(index=["low", "medium", "high"], columns=["low", "medium", "high"], fill_value=0)

group_metrics = []
for group_name, group_df in results_df.groupby("overlap_bucket", sort=True):
    if len(group_df) >= 2:
        group_pearson = pearsonr(group_df["predicted_score_0_5"], group_df["label"]).statistic
        group_spearman = spearmanr(group_df["predicted_score_0_5"], group_df["label"]).statistic
    else:
        group_pearson = np.nan
        group_spearman = np.nan
    group_metrics.append({
        "overlap_bucket": group_name,
        "n": int(len(group_df)),
        "label_mean": float(group_df["label"].mean()),
        "pred_mean": float(group_df["predicted_score_0_5"].mean()),
        "mae": float(group_df["absolute_error"].mean()),
        "pearson": None if pd.isna(group_pearson) else float(group_pearson),
        "spearman": None if pd.isna(group_spearman) else float(group_spearman),
        "band_accuracy": float(group_df["band_match"].mean()),
    })

group_metrics_df = pd.DataFrame(group_metrics).sort_values("overlap_bucket").reset_index(drop=True)

print(results_df[["sentence1", "sentence2", "label", "lexical_jaccard", "cosine_similarity", "predicted_score_0_5", "absolute_error", "label_band", "pred_band"]].head(10))
print(group_metrics_df)
print(confusion_counts)
print(high_overlap_errors_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "lexical_jaccard", "shared_content_words", "label_band", "pred_band"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

overall_mae = float(results_df["absolute_error"].mean())
band_accuracy = float(results_df["band_match"].mean())

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"original_num_examples: {len(df)}")
print(f"base_num_examples: {len(base_df)}")
print(f"subset_num_examples: {len(subset_df)}")
print("subset_rule: hard_negatives_label<=2_with_high_lexical_overlap_and_hard_positives_label>=4_with_low_lexical_overlap_deterministic_top_150_each_then_concat_dedup")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mean_absolute_error: {overall_mae:.6f}")
print(f"band_accuracy_low_medium_high: {band_accuracy:.6f}")
print(f"avg_lexical_jaccard: {results_df['lexical_jaccard'].mean():.6f}")
print(f"avg_shared_content_words: {results_df['shared_content_words'].mean():.6f}")
print(f"num_high_overlap_examples: {(results_df['overlap_bucket'] == 'high_overlap').sum()}")
print(f"num_low_overlap_examples: {(results_df['overlap_bucket'] == 'low_overlap').sum()}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

print(confusion_counts.to_dict())
print(group_metrics_df.to_dict(orient="records"))

top_misleading_high_overlap = high_overlap_errors_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "lexical_jaccard", "shared_content_words", "label_band", "pred_band"
]].head(5)
print(top_misleading_high_overlap.to_dict(orient="records"))
